In [9]:
import pandas as pd
import networkx as nx
from pathlib import Path
import numpy as np

# Go up one folder from code/ and into output/
output = Path.cwd().parent / "output"

edges = pd.read_csv(output / "airport_edges_2025.csv")
attrs = pd.read_csv(output / "airport_attributes_2025.csv", index_col="airport")

G = nx.from_pandas_edgelist(
    edges,
    "origin",
    "dest",
    edge_attr="passengers",
    create_using=nx.DiGraph
)

nx.set_node_attributes(G, attrs.to_dict("index"))

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 576
Edges: 8156


In [16]:
top_20_measles = attrs.sort_values(
    by="cases_2025",
    ascending=False
).head(20)

print(top_20_measles[["name", "cases_2025", "first_case_date"]])

                                                      name  cases_2025  \
airport                                                                  
HOB                            Lea County Regional Airport         501   
GSP           Greenville-Spartanburg International Airport         330   
IFP                Laughlin Bullhead International Airport         216   
SGU                             St George Regional Airport         155   
LBB            Lubbock Preston Smith International Airport         144   
ELP                          El Paso International Airport          61   
MAF               Midland International Air and Space Port          52   
GCK                           Garden City Regional Airport          48   
GRB                  Austin Straubel International Airport          36   
TYR                          Tyler Pounds Regional Airport          29   
PVU                                Provo Municipal Airport          25   
ERI                     Erie Internati

In [12]:
bc = nx.betweenness_centrality(G)

top_5_bc = sorted(bc.items(), key=lambda x: x[1], reverse=True)[:5]

print(top_5_bc)

ec = nx.eigenvector_centrality(G)

top_5_ec = sorted(ec.items(), key=lambda x: x[1], reverse=True)[:5]
print(top_5_ec)

cc = nx.closeness_centrality(G)

top_5_cc = sorted(cc.items(), key=lambda x: x[1], reverse=True)[:5]
print(top_5_cc)

[('ANC', 0.2856736265597117), ('DEN', 0.1470301486453682), ('ORD', 0.12827395420561713), ('DFW', 0.12132642062806134), ('BET', 0.10783669141039237)]
[('DFW', 0.16943720138283078), ('ATL', 0.16795066662617922), ('DEN', 0.16696712868685876), ('ORD', 0.1663288232017705), ('CLT', 0.15402363094634228)]
[('DEN', 0.5307082201086957), ('ORD', 0.5281294629653104), ('DFW', 0.5200432702309132), ('ATL', 0.5045916596019538), ('MSP', 0.5008711681025846)]


In [10]:
# Add centrality scores to the graph
nx.set_node_attributes(G, bc, "betweenness")

# Create a dataframe combining airport and measles data
results = pd.DataFrame.from_dict(
    nx.get_node_attributes(G, "cases_2025"),
    orient="index",
    columns=["cases_2025"]
)

results["betweenness"] = pd.Series(bc)

# Add first case date
results["first_case_date"] = pd.Series(
    nx.get_node_attributes(G, "first_case_date")
)

# Display top 5 airports by betweenness
print(results.sort_values("betweenness", ascending=False).head(5))

     cases_2025  betweenness first_case_date
ANC           0     0.285674             NaN
DEN          13     0.147030      2025-04-07
ORD           0     0.128274             NaN
DFW           6     0.121326      2025-05-02
BET           0     0.107837             NaN


In [13]:
# Add centrality scores to the graph
nx.set_node_attributes(G, bc, "eigenvector")

# Create a dataframe combining airport and measles data
results = pd.DataFrame.from_dict(
    nx.get_node_attributes(G, "cases_2025"),
    orient="index",
    columns=["cases_2025"]
)

results["eigenvector"] = pd.Series(ec)

# Add first case date
results["first_case_date"] = pd.Series(
    nx.get_node_attributes(G, "first_case_date")
)

# Display top 5 airports by betweenness
print(results.sort_values("eigenvector", ascending=False).head(5))

     cases_2025  eigenvector first_case_date
DFW           6     0.169437      2025-05-02
ATL          10     0.167951      2025-01-28
DEN          13     0.166967      2025-04-07
ORD           0     0.166329             NaN
CLT           0     0.154024             NaN
